# End-to-End Sentiment Analysis Pipeline (Including BERT)


This notebook combines data processing, baseline models (TF-IDF & Word2Vec with Logistic Regression), an Advanced Model (DistilBERT), and an interactive Gradio UI.


## 1. Installations and Imports


In [1]:
!pip install -q pandas numpy scikit-learn gensim gradio transformers torch

import pandas as pd
import numpy as np
import re
import pickle
import os
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import gensim
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import gradio as gr
import warnings
warnings.filterwarnings('ignore')

## 2. Data Loading and Preprocessing


In [2]:
# Load Data
data = pd.read_csv('/kaggle/input/datasets/dongrelaxman/amazon-reviews-dataset/Amazon_Reviews.csv', engine='python')


# Preprocessing for baseline models
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

if 'Rating' in data.columns:
    data['Rating'] = data['Rating'].astype(str).str.extract(r'(\d)').astype(float)
    data = data.dropna(subset=['Rating'])
    data['Sentiment'] = data['Rating'].apply(lambda x: 1 if x >= 4 else (0 if x <= 2 else None))
    data = data.dropna(subset=['Sentiment'])
    data['Sentiment'] = data['Sentiment'].astype(int)
    
if 'Review Title' in data.columns and 'Review Text' in data.columns:
    data['Full_Review'] = data['Review Title'].fillna('') + " " + data['Review Text'].fillna('')
elif 'Review Text' in data.columns:
    data['Full_Review'] = data['Review Text'].fillna('')
else:
    data['Full_Review'] = data.iloc[:, 0].astype(str)

data['Clean_Review'] = data['Full_Review'].apply(preprocess_text)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(data['Clean_Review'], data['Sentiment'], test_size=0.2, random_state=42)
X_train_raw, X_test_raw, _, _ = train_test_split(data['Full_Review'], data['Sentiment'], test_size=0.2, random_state=42)

print(f"Training samples: {len(X_train)}, Testing samples: {len(X_test)}")

Training samples: 16136, Testing samples: 4034


## 3. Baseline Models: Feature Extraction & Training


In [7]:
# --- TF-IDF ---
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

lr_tfidf = LogisticRegression(max_iter=1000)
lr_tfidf.fit(X_train_tfidf, y_train)

# --- Word2Vec ---
sentences_train = [text.split() for text in X_train]
sentences_test = [text.split() for text in X_test]
w2v_model = gensim.models.Word2Vec(sentences_train, vector_size=100, window=5, min_count=1, workers=4)

def get_sentence_vector(words, model):
    valid_words = [word for word in words if word in model.wv]
    if valid_words:
        return np.mean(model.wv[valid_words], axis=0)
    else:
        return np.zeros(model.vector_size)

X_train_w2v = np.array([get_sentence_vector(words, w2v_model) for words in sentences_train])
X_test_w2v = np.array([get_sentence_vector(words, w2v_model) for words in sentences_test])

lr_w2v = LogisticRegression(max_iter=1000)
lr_w2v.fit(X_train_w2v, y_train)

print("Baseline Models Trained!")

Baseline Models Trained!


## 4. Advanced Model: BERT (DistilBERT)


In [18]:
# Initialize Tokenizer and Model
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model correctly for classification task
bert_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    ignore_mismatched_sizes=True  # مهم لتفادي warnings زي اللي ظهرت عندك
)

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bert_model.to(device)

print(f"BERT Model loaded on {device}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT Model loaded on cpu


## 5. Saving Models


In [19]:
os.makedirs("models", exist_ok=True)

# Save Baseline Models
with open("models/tfidf_vectorizer.pkl", "wb") as f: pickle.dump(tfidf_vectorizer, f)
with open("models/lr_tfidf.pkl", "wb") as f: pickle.dump(lr_tfidf, f)
w2v_model.save("models/w2v_model.gensim")
with open("models/lr_w2v.pkl", "wb") as f: pickle.dump(lr_w2v, f)

# Save BERT Model
tokenizer.save_pretrained("models/distilbert_sentiment")
bert_model.save_pretrained("models/distilbert_sentiment")
print("All models saved successfully.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

All models saved successfully.


## 6. Interactive Gradio Interface (Blocks)


In [20]:
# Inference Function
def predict_sentiment(text, model_type, feature_ext):
    if model_type == "Advanced (BERT)":
        # BERT Inference
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
        with torch.no_grad():
            outputs = bert_model(**inputs)
            logits = outputs.logits
            pred = torch.argmax(logits, dim=1).item()
    else:
        # Baseline Inference
        clean_text = preprocess_text(text)
        if feature_ext == "TF-IDF":
            vec = tfidf_vectorizer.transform([clean_text])
            pred = lr_tfidf.predict(vec)[0]
        else: # Word2Vec
            words = clean_text.split()
            vec = get_sentence_vector(words, w2v_model).reshape(1, -1)
            pred = lr_w2v.predict(vec)[0]
            
    return "Positive 🟢" if pred == 1 else "Negative 🔴"

# Gradio Blocks UI
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🤖 Sentiment Analysis Classifier")
    gr.Markdown("Choose between our Baseline Models (Logistic Regression) or our Advanced Model (DistilBERT).")
    
    with gr.Row():
        with gr.Column():
            text_in = gr.Textbox(lines=4, placeholder="Enter a review here...", label="Review Text")
            model_choice = gr.Radio(["Baseline", "Advanced (BERT)"], value="Baseline", label="Select Model Type")
            
            # This radio will be dynamically hidden when BERT is selected
            feat_choice = gr.Radio(["TF-IDF", "Word2Vec"], value="TF-IDF", label="Feature Extraction (Baseline Only)")
            
            predict_btn = gr.Button("Analyze Sentiment", variant="primary")
            
        with gr.Column():
            output_text = gr.Textbox(label="Predicted Sentiment", text_align="center")

    # Dynamic visibility logic
    def update_visibility(choice):
        if choice == "Advanced (BERT)":
            return gr.update(visible=False)
        else:
            return gr.update(visible=True)
            
    model_choice.change(fn=update_visibility, inputs=model_choice, outputs=feat_choice)
    
    # Prediction click
    predict_btn.click(fn=predict_sentiment, inputs=[text_in, model_choice, feat_choice], outputs=output_text)

# Launch
demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://f16d478009ea0ad31a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
